# 20 — Roadmap item 7: flip TTA on the augment checkpoints

**Why only the augment checkpoints** (`project_dat_parkinson_strategic_roadmap.md`,
item 7): EDA section 6b (`README.md`) established the discriminative signal
(striatal L-R asymmetry *magnitude*) is flip-invariant, so L-R flip TTA is
principled here. But `train_one_fold` was never told about flips for
rung3/familybias/lrsched/classweight/fixedepoch -- those checkpoints are
NOT flip-equivariant, and flipping their input at test time is untested and
could easily hurt rather than help. Only `rung4_augment_*` (trained with
`random_flip`, `src/augment.py`, `L_R_AXIS=1`) saw flipped inputs during
training and is the principled TTA candidate.

**Method**: for each of the augment family's 25 checkpoints (5 seeds x 5
folds), run inference on that fold's outer-test volumes twice -- once
plain (reproduces the existing `rung4_augment_oof_seed{s}.npy`, checked as
a sanity check) and once with a deterministic L-R flip
(`np.flip(volume, axis=augment.L_R_AXIS)`, the same axis convention
`src/augment.py::random_flip` already uses) -- then average the two
probabilities per row. GPU required but inference-only, no training, so
much faster than notebooks 09-13/18.

**Pre-registered comparison (same discipline as notebooks 16/18)**: does
swapping the augment variant's plain OOF for its flip-TTA'd OOF inside the
already-adopted 6-variant ensemble improve the ensemble metric (current
adopted composition, notebook 18: log loss=0.3978, AUROC=0.9004,
ECE=0.0305)? One comparison, no further searching.

**Known cost if adopted for real**: TTA roughly doubles in-container
inference time for the augment checkpoints specifically (25 of the
ensemble's 150) -- worth keeping in mind against the 3-hour submission
budget, but not a reason to skip validating it locally first.

**Data handling**: loads real row-level labels and runs inference on real
`.nii.gz` volumes (via the shared cache), so per the AI-assistant data
rule (`README.md`) this is **[RUN ME]** — run it yourself, share back only
the printed aggregate numbers.

In [1]:
# [RUN ME] -- loads real pixel data + row-level labels. Reuses the shared
# on-disk volume cache (same preprocessing as the augment checkpoints were
# trained on).
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import augment
import cache
import config
import dataset
import evaluate
import model

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()
y_true = np.array(labels)

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
print(f"cache {'reused' if volume_cache.was_reused else 'rebuilt'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")


def flip_lr(volume):
    return np.flip(volume, axis=augment.L_R_AXIS).copy().astype(np.float32)

cache reused in 0.4s for 1362 volumes


In [2]:
# [RUN ME] -- inference-only (no training) over the 25 already-trained
# augment checkpoints: plain pass (sanity-checked against the saved OOF)
# + flipped pass, per fold. GPU, but much faster than a training notebook.
repeat_seeds = list(range(config.SEED, config.SEED + 5))
batch_size = 32
inference_start = time.time()

oof_augment_original = []
oof_augment_flipped = []
oof_augment_tta = []

for repeat_seed in repeat_seeds:
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    orig_probs = np.zeros(len(uids))
    flip_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_test_uids = [uids[i] for i in test_idx]
        checkpoint_path = config.CHECKPOINT_DIR / f"rung4_augment_seed{repeat_seed}_fold{fold_i}.pt"
        net = model.build_model().to(config.DEVICE)
        net.load_state_dict(torch.load(checkpoint_path, map_location=config.DEVICE))

        orig_ds = dataset.DatParkinsonDataset(fold_test_uids, load_fn=volume_cache.get)
        orig_loader = torch.utils.data.DataLoader(orig_ds, batch_size=batch_size, num_workers=0)
        orig_probs[test_idx] = np.concatenate([model.predict(net, x) for x, _ in orig_loader])

        flip_ds = dataset.DatParkinsonDataset(fold_test_uids, load_fn=volume_cache.get, transform=flip_lr)
        flip_loader = torch.utils.data.DataLoader(flip_ds, batch_size=batch_size, num_workers=0)
        flip_probs[test_idx] = np.concatenate([model.predict(net, x) for x, _ in flip_loader])

    oof_augment_original.append(orig_probs)
    oof_augment_flipped.append(flip_probs)
    tta_probs = (orig_probs + flip_probs) / 2
    oof_augment_tta.append(tta_probs)

    saved_oof = np.load(config.DATA_PROCESSED / f"rung4_augment_oof_seed{repeat_seed}.npy")
    sanity_diff = np.abs(orig_probs - saved_oof).max()
    print(f"seed={repeat_seed}: sanity check (plain vs. saved OOF) max abs diff={sanity_diff:.6f} -- "
          f"plain={evaluate.log_loss_score(y_true, orig_probs):.4f}, "
          f"flip-only={evaluate.log_loss_score(y_true, flip_probs):.4f}, "
          f"TTA(avg)={evaluate.log_loss_score(y_true, tta_probs):.4f}")

print(f"\ninference time: {(time.time() - inference_start) / 60:.1f} min")

seed=42: sanity check (plain vs. saved OOF) max abs diff=0.000000 -- plain=0.4647, flip-only=0.4617, TTA(avg)=0.4607
seed=43: sanity check (plain vs. saved OOF) max abs diff=0.000000 -- plain=0.4431, flip-only=0.4439, TTA(avg)=0.4423
seed=44: sanity check (plain vs. saved OOF) max abs diff=0.000000 -- plain=0.4205, flip-only=0.4148, TTA(avg)=0.4148
seed=45: sanity check (plain vs. saved OOF) max abs diff=0.000000 -- plain=0.4400, flip-only=0.4359, TTA(avg)=0.4359
seed=46: sanity check (plain vs. saved OOF) max abs diff=0.000000 -- plain=0.4471, flip-only=0.4482, TTA(avg)=0.4454

inference time: 0.2 min


In [3]:
# [RUN ME] (no new data access -- uses arrays built above + disk).
# THE single pre-registered comparison: swap augment's plain OOF for its
# flip-TTA'd OOF inside the already-adopted 6-variant ensemble. No further
# searching (e.g. flip-only, or TTA on other variants) feeds this decision.
current_variant_prefixes = ["rung3", "rung4_familybias", "rung4_lrsched",
                             "rung4_classweight", "rung4_fixedepoch"]  # augment handled separately below

non_augment_arrays = [
    np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in current_variant_prefixes for s in repeat_seeds
]
plain_augment_arrays = [np.load(config.DATA_PROCESSED / f"rung4_augment_oof_seed{s}.npy") for s in repeat_seeds]

baseline_ensemble_oof = np.mean(non_augment_arrays + plain_augment_arrays, axis=0)   # current adopted composition
tta_ensemble_oof = np.mean(non_augment_arrays + oof_augment_tta, axis=0)             # augment swapped for its TTA'd version

baseline_scores = evaluate.combined_score(y_true, baseline_ensemble_oof)
tta_scores = evaluate.combined_score(y_true, tta_ensemble_oof)

print(f"6-variant ensemble, plain augment (current):     log loss={baseline_scores['log_loss']:.4f}  "
      f"AUROC={baseline_scores['auroc']:.4f}  ECE={baseline_scores['ece']:.4f}")
print(f"6-variant ensemble, augment swapped for flip-TTA: log loss={tta_scores['log_loss']:.4f}  "
      f"AUROC={tta_scores['auroc']:.4f}  ECE={tta_scores['ece']:.4f}")
print(f"\ndelta (TTA - plain): {tta_scores['log_loss'] - baseline_scores['log_loss']:+.4f} log loss")
print("THE DECISION RULE: if this delta is negative, adopt flip-TTA for the augment checkpoints "
      "(then re-run the calibration+blend re-tuning against this composition). If positive/flat, "
      "discard TTA -- keep the plain augment OOF, no further searching (flip-only, other variants, etc.).")

6-variant ensemble, plain augment (current):     log loss=0.3978  AUROC=0.9004  ECE=0.0305
6-variant ensemble, augment swapped for flip-TTA: log loss=0.3976  AUROC=0.9005  ECE=0.0293

delta (TTA - plain): -0.0002 log loss
THE DECISION RULE: if this delta is negative, adopt flip-TTA for the augment checkpoints (then re-run the calibration+blend re-tuning against this composition). If positive/flat, discard TTA -- keep the plain augment OOF, no further searching (flip-only, other variants, etc.).


In [4]:
# [RUN ME] -- no new data access, persists the arrays already computed
# above in this kernel session. Decision (per the pre-registered rule,
# delta=-0.0002 negative): adopt flip-TTA for the augment checkpoints.
# Saves them under a distinct prefix so the plain rung4_augment_oof
# arrays stay on disk too (nothing is overwritten/lost).
for repeat_seed, tta_probs in zip(repeat_seeds, oof_augment_tta):
    np.save(config.DATA_PROCESSED / f"rung4_augment_tta_oof_seed{repeat_seed}.npy", tta_probs)
print(f"saved rung4_augment_tta_oof_seed{{{repeat_seeds[0]}..{repeat_seeds[-1]}}}.npy")

saved rung4_augment_tta_oof_seed{42..46}.npy


**What we're looking for:** does averaging each augment checkpoint's
prediction with its L-R-flipped counterpart (principled here specifically
because these checkpoints were trained with flip augmentation) improve the
already-adopted 6-variant ensemble?

**What we found:**
```
seed=42: plain=0.4647, flip-only=0.4617, TTA(avg)=0.4607
seed=43: plain=0.4431, flip-only=0.4439, TTA(avg)=0.4423
seed=44: plain=0.4205, flip-only=0.4148, TTA(avg)=0.4148
seed=45: plain=0.4400, flip-only=0.4359, TTA(avg)=0.4359
seed=46: plain=0.4471, flip-only=0.4482, TTA(avg)=0.4454

6-variant ensemble, plain augment (current):      log loss=0.3978  AUROC=0.9004  ECE=0.0305
6-variant ensemble, augment swapped for flip-TTA: log loss=0.3976  AUROC=0.9005  ECE=0.0293

delta (TTA - plain): -0.0002 log loss
```
Two independent signals both point the same way: at the augment-variant
level, TTA(avg) beats plain in **5/5 seeds** individually (not just on
average); at the ensemble level, log loss and ECE both improve (-0.0002,
-0.0012) while AUROC is essentially flat (+0.0001) -- directionally
coherent with TTA acting mainly as a calibration smoother (averaging two
correlated views narrows the predictive distribution) rather than adding
new discriminative signal, which matches this roadmap step's own
expectation. The ensemble-level delta is small -- smaller than notebooks
5/18's own deltas -- because augment is only 1 of 6 ensemble members, but
the 5/5 per-seed unanimity at the variant level is the stronger piece of
evidence that this isn't noise. Per the pre-registered rule below, the
decision is mechanical regardless of the delta's size.

**Decision / next step:** delta is negative → **adopt flip-TTA for the
augment checkpoints** in the final composition. Next: `notebooks/21`
re-runs the calibration+blend re-tuning (notebook 19's exact method)
against this composition -- the augment family's per-seed OOF is now its
flip-TTA'd version, `rung4_augment_tta_oof_seed{42..46}.npy` (saved by the
cell above), in place of the plain `rung4_augment_oof_seed{s}.npy`. Params
will likely shift slightly again. Per the roadmap
(`project_dat_parkinson_strategic_roadmap.md`), item 7 is now the last
open local-CV item -- after notebook 21, all of items 1-7 are addressed
and the next step is implementation: wire the adopted recipe into
`src/submission.py::combine_predictions` and `submission_src/main.py`
(now needs both the flip-TTA inference path and the 6-variant
composition), extend the checkpoint-list packaging, address the
sklearn-pickle risk, rebuild + smoke-test, before spending a real
submission. Keep in mind flip-TTA roughly doubles in-container inference
time for the augment checkpoints specifically (25 of 150) -- recheck the
3-hour submission budget once the full recipe is assembled.